# 00 · Data Overview

Notebook base para entender el dataset, revisar la variable objetivo y dejar preparadas las decisiones de modelado comparables entre enfoques espaciales y no lineales.

## Objetivos

- Revisar calidad y cobertura de datos.
- Inspeccionar la distribución de `precio_m2` o su transformación.
- Explorar patrones espaciales iniciales.
- Detectar variables candidatas y potenciales sesgos del dataset.
- Dejar documentadas hipótesis para los notebooks de modelado.

In [17]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

OUTPUT_DIR = PROJECT_ROOT / "notebooks" / "output" / "00_data_overview"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 200)
PROJECT_ROOT

PosixPath('/home/saneliges/Escritorio/PredictorPrecioMetroCuadradoAlquileres')

## Carga de datos

Se cargan los datos procesados de `data/processed`: alquiler (`arg_alquiler_caba_processed.csv`) y venta (`arg_venta_caba_processed.csv`). El análisis principal se enfoca en el dataset de venta.

Resumen rápido:
- `arg_alquiler_caba_processed.csv`: 989 filas x 31 columnas.
- `arg_venta_caba_processed.csv`: 10614 filas x 31 columnas.
- Objetivo base para modelado: `precio_sobre_m2`.
- Coordenadas espaciales: `longitud`, `latitud`.
- Variables con nulos importantes: `tipo_unidad`, `antiguedad`, `estado`, `disposicion`, `orientacion`, `estado_edificio`.


In [18]:
DATA_DIR = PROJECT_ROOT / "data" / "processed"
rental_path = DATA_DIR / "arg_alquiler_caba_processed.csv"
sale_path = DATA_DIR / "arg_venta_caba_processed.csv"

with pd.option_context("display.max_columns", 200):
    df_rental = pd.read_csv(rental_path)
    df_sale = pd.read_csv(sale_path)

# Dataset principal para análisis de venta.
df = df_sale

target_col = "precio_sobre_m2"
coord_cols = ["longitud", "latitud"]
id_col = "id"

print("Dataset de alquileres:", df_rental.shape)
print("Dataset de venta:", df_sale.shape)
print("Columnas comunes entre ambos datasets:")
print(sorted(set(df_rental.columns) & set(df_sale.columns)))

print("Primeras filas del dataset de venta:")
display(df_sale.head(3))


Dataset de alquileres: (989, 31)
Dataset de venta: (10614, 31)
Columnas comunes entre ambos datasets:
['ambientes', 'antiguedad', 'area_m2_cubierta', 'area_m2_descubierta', 'area_m2_total', 'banos', 'barrio', 'cocheras', 'comuna', 'disposicion', 'dormitorios', 'estado', 'estado_edificio', 'estado_num', 'expensas', 'id', 'idx', 'image_url', 'imagen_path', 'index_right', 'latitud', 'log_precio', 'longitud', 'moneda', 'orientacion', 'precio', 'precio_sobre_m2', 'tipo_unidad', 'url', 'valido_desde', 'valido_hasta']
Primeras filas del dataset de venta:


,idx,id,url,precio,moneda,expensas,tipo_unidad,area_m2_cubierta,area_m2_descubierta,area_m2_total,antiguedad,estado_edificio,ambientes,banos,estado,disposicion,orientacion,latitud,longitud,image_url,imagen_path,valido_desde,valido_hasta,cocheras,dormitorios,index_right,barrio,comuna,estado_num,log_precio,precio_sobre_m2
0,4257,18875423.0,https://www.argenprop.com/departamento-en-vent...,70000.0,USD,113400.0,NaN,34.0,0.00,34.0,51.0,NaN,2.0,1.0,Muy Bueno,Lateral,NaN,-34.59357,-58.44511,https://www.argenprop.com/static-content/32457...,images/18875423.jpg,2026-02-05 15:18:35.864925,NaN,0.0,NaN,35.0,Villa Crespo,15.0,4.0,11.156251,2058.823529
1,4262,18875458.0,https://www.argenprop.com/departamento-en-vent...,284000.0,USD,0.0,Duplex,123.0,49.54,196.0,2.0,NaN,4.0,2.0,Excelente,Frente,NaN,-34.61034,-58.49715,https://www.argenprop.com/static-content/85457...,images/18875458.jpg,2026-02-05 15:19:02.071369,NaN,0.0,NaN,36.0,Villa Del Parque,11.0,5.0,12.556730,1448.979592
2,4265,18172428.0,https://www.argenprop.com/departamento-en-vent...,125000.0,USD,230000.0,NaN,84.0,0.00,84.0,44.0,NaN,4.0,1.0,Bueno,NaN,NaN,-34.61421,-58.43436,https://www.argenprop.com/static-content/82427...,images/18172428.jpg,2026-02-05 15:19:16.333598,NaN,0.0,NaN,7.0,Caballito,6.0,3.0,11.736069,1488.095238


## Sanity checks iniciales

In [19]:
for name, dataset in [("Alquiler", df_rental), ("Venta", df_sale)]:
    print(f"--- {name} ---")
    print("shape:", dataset.shape)
    print("columnas:", len(dataset.columns))
    print("nulos por columna (%):")
    display((dataset.isna().mean().sort_values(ascending=False) * 100).head(20))
    print("\nTipos de datos:")
    display(dataset.dtypes.sort_index())
    print("\n")

print("Dataset principal para análisis inicial: venta")
print("Columnas objetivo y espaciales:")
print("target_col:", target_col)
print("coord_cols:", coord_cols)
print("id_col:", id_col)

print("Resumen estadístico rápido del dataset de venta:")
display(df.describe(include="all").T.head(20))


--- Alquiler ---
shape: (989, 31)
columnas: 31
nulos por columna (%):


valido_hasta           100.000000
estado_edificio        100.000000
orientacion             56.521739
dormitorios             45.702730
antiguedad              26.592518
estado                  25.480283
estado_num              25.480283
tipo_unidad             22.548028
disposicion             12.740142
imagen_path              0.303337
precio                   0.000000
moneda                   0.000000
valido_desde             0.000000
id                       0.000000
idx                      0.000000
ambientes                0.000000
area_m2_total            0.000000
area_m2_cubierta         0.000000
area_m2_descubierta      0.000000
expensas                 0.000000
dtype: float64


Tipos de datos:


ambientes              float64
antiguedad             float64
area_m2_cubierta       float64
area_m2_descubierta    float64
area_m2_total          float64
banos                  float64
barrio                     str
cocheras               float64
comuna                 float64
disposicion                str
dormitorios            float64
estado                     str
estado_edificio        float64
estado_num             float64
expensas               float64
id                     float64
idx                      int64
image_url                  str
imagen_path                str
index_right            float64
latitud                float64
log_precio             float64
longitud               float64
moneda                     str
orientacion                str
precio                 float64
precio_sobre_m2        float64
tipo_unidad                str
url                        str
valido_desde               str
valido_hasta           float64
dtype: object



--- Venta ---
shape: (10614, 31)
columnas: 31
nulos por columna (%):


estado_edificio        100.000000
valido_hasta           100.000000
orientacion             96.221971
tipo_unidad             94.874694
estado_num              93.094027
estado                  93.094027
disposicion             93.075184
imagen_path             75.522894
dormitorios             22.131148
antiguedad              13.604673
image_url                0.028265
expensas                 0.000000
url                      0.000000
id                       0.000000
idx                      0.000000
banos                    0.000000
ambientes                0.000000
area_m2_descubierta      0.000000
area_m2_total            0.000000
area_m2_cubierta         0.000000
dtype: float64


Tipos de datos:


ambientes              float64
antiguedad             float64
area_m2_cubierta       float64
area_m2_descubierta    float64
area_m2_total          float64
banos                  float64
barrio                     str
cocheras               float64
comuna                 float64
disposicion                str
dormitorios            float64
estado                     str
estado_edificio        float64
estado_num             float64
expensas               float64
id                     float64
idx                      int64
image_url                  str
imagen_path                str
index_right            float64
latitud                float64
log_precio             float64
longitud               float64
moneda                     str
orientacion                str
precio                 float64
precio_sobre_m2        float64
tipo_unidad                str
url                        str
valido_desde               str
valido_hasta           float64
dtype: object



Dataset principal para análisis inicial: venta
Columnas objetivo y espaciales:
target_col: precio_sobre_m2
coord_cols: ['longitud', 'latitud']
id_col: id
Resumen estadístico rápido del dataset de venta:


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
idx,10614.0,NaN,NaN,NaN,27886.740814,10132.587469,4257.0,26328.25,30785.5,34883.5,40131.0
id,10614.0,NaN,NaN,NaN,17743437.970228,2439051.929244,2755601.0,17661406.0,18619085.5,19117193.0,19351416.0
url,10614,10614,https://www.argenprop.com/departamento-en-vent...,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
precio,10614.0,NaN,NaN,NaN,161428.435557,168874.51752,1.0,78000.0,120000.0,182216.75,3500000.0
moneda,10614,2,USD,10593,NaN,NaN,NaN,NaN,NaN,NaN,NaN
expensas,10614.0,NaN,NaN,NaN,121511.432353,207921.879125,0.0,0.0,80000.0,170000.0,10300000.0
tipo_unidad,544,9,Departamento,484,NaN,NaN,NaN,NaN,NaN,NaN,NaN
area_m2_cubierta,10614.0,NaN,NaN,NaN,58.608722,54.508205,0.0,35.455,48.0,70.0,3667.0
area_m2_descubierta,10614.0,NaN,NaN,NaN,6.717142,17.83466,0.0,0.0,0.0,6.0,583.0
area_m2_total,10614.0,NaN,NaN,NaN,107.91624,2102.644485,1.0,40.0,56.0,82.0,149000.0


## Variable objetivo

Explorá tanto la escala original como una posible transformación logarítmica si después la vas a usar en todos los modelos.

In [20]:
# fig, axes = plt.subplots(1, 2, figsize=(14, 5))
# sns.histplot(df[target_col], kde=True, ax=axes[0])
# axes[0].set_title("Distribucion de precio_m2")
# sns.histplot(np.log1p(df[target_col]), kde=True, ax=axes[1])
# axes[1].set_title("Distribucion de log1p(precio_m2)")
# plt.tight_layout()

## Exploración espacial

Acá conviene mirar mapas simples, densidad por zona y gradientes espaciales del target.

In [21]:
# import geopandas as gpd
# gdf = gpd.GeoDataFrame(df.copy(), geometry=gpd.points_from_xy(df[coord_cols[0]], df[coord_cols[1]]), crs="EPSG:4326")
# ax = gdf.plot(column=target_col, figsize=(9, 9), alpha=0.7, markersize=8, legend=True)
# ax.set_title("Precio por m2 en el espacio")
# ax.set_axis_off()

## Correlaciones y señales candidatas

In [22]:
# numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
# corr = df[numeric_cols].corr(numeric_only=True)[target_col].sort_values(ascending=False)
# display(corr.head(20))
# display(corr.tail(20))

## Preview del protocolo de split

Documentá si el split será aleatorio, temporal, espacial o híbrido. Este punto es clave para que la comparación entre modelos sea justa.

In [23]:
# Ejemplos de checks previos al split.
# df["fecha_publicacion"] = pd.to_datetime(df["fecha_publicacion"])
# display(df["fecha_publicacion"].agg(["min", "max"]))
# display(df["barrio"].value_counts().head(15))

## Hallazgos e hipótesis

- ¿Qué señales parecen más explicativas?
- ¿Dónde hay heterogeneidad espacial visible?
- ¿Qué outliers conviene tratar antes del modelado?
- ¿Qué variables deberían estar sí o sí en todos los modelos?